In [2]:
# ===========================
# CELLULE 1 : Import des librairies
# ===========================
import boto3               # Pour accéder aux fichiers sur Amazon S3
import pandas as pd
import json                # Pour lire les fichiers JSONL
from pymongo import MongoClient
import re                  # Pour nettoyer les chaînes de caractères
import os


print(" Librairies importées avec succès")

# ===========================
# CELLULE 2 : Configuration S3
# ===========================
BUCKET_NAME = "greencoop-forecast2-raw"
REGION_NAME = "eu-west-3"

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

print("AWS_ACCESS_KEY_ID chargé :", AWS_ACCESS_KEY_ID is not None)
print("AWS_SECRET_ACCESS_KEY chargé :", AWS_SECRET_ACCESS_KEY is not None)

# Création du client S3 pour accéder aux fichiers
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=REGION_NAME
)
print(f" Connexion S3 établie pour le bucket {BUCKET_NAME}")

# ===========================
# CELLULE 3 : Fonctions pour lire les fichiers depuis S3
# ===========================
def read_jsonl_from_s3(s3_path):
    print(f" Lecture du fichier JSONL : {s3_path}")
    response = s3.get_object(Bucket=BUCKET_NAME, Key=s3_path)
    data = response['Body'].read().decode('utf-8').splitlines()
    records = [json.loads(line)['_airbyte_data'] for line in data]
    df = pd.DataFrame(records)
    print(f" Lecture terminée : {len(df)} lignes et {len(df.columns)} colonnes")
    return df

def read_excel_from_s3(s3_path):
    print(f" Lecture du fichier Excel : {s3_path}")
    response = s3.get_object(Bucket=BUCKET_NAME, Key=s3_path)
    df = pd.read_excel(response['Body'])
    print(f" Lecture terminée : {len(df)} lignes et {len(df.columns)} colonnes")
    return df

# ===========================
# CELLULE 4 : Définir les chemins S3
# ===========================
s3_paths = {
    "InfoClimat": "data_sync/InfoClimat_France/2026_01_12_1768220353795_0.jsonl",
    "WeatherIchtegem": "data_sync/weather_iichtegem/2026_01_12_1768224585495_0.jsonl",
    "WeatherLaMadeleine": "data_sync/WeatherUnderground_LaMadeleine/2026_01_12_1768226046168_0.jsonl"
}

# ===========================
# CELLULE 5 : Lecture des données
# ===========================
df_infoclimat = read_jsonl_from_s3(s3_paths["InfoClimat"])
df_ichtegem = read_jsonl_from_s3(s3_paths["WeatherIchtegem"])
df_lamadeleine = read_jsonl_from_s3(s3_paths["WeatherLaMadeleine"])

print("\n Aperçu des données chargées :")
print("InfoClimat :", df_infoclimat.shape)
print("Weather Ichtegem :", df_ichtegem.shape)
print("Weather La Madeleine :", df_lamadeleine.shape)

# ===========================
# CELLULE 6 : Nettoyage et transformation des données météo
# ===========================
def clean_weather_data(df):
    print("\n Début du nettoyage et transformation des données météo")

    # Colonnes de températures
    for col in ['Temperature', 'Dew Point']:
        if col in df.columns:
            df[col] = df[col].astype(str).apply(lambda x: float(re.sub(r'[^\d.]', '', x)))
            print(f"   Colonne '{col}' convertie en float")

    # Humidité
    if 'Humidity' in df.columns:
        df['Humidity'] = df['Humidity'].astype(str).apply(lambda x: float(re.sub(r'[^\d.]', '', x)))
        print(f"   Colonne 'Humidity' convertie en float")

    # Vitesse du vent
    for col in ['Speed', 'Gust']:
        if col in df.columns:
            df[col] = df[col].astype(str).apply(lambda x: float(re.sub(r'[^\d.]', '', x)))
            print(f"   Colonne '{col}' convertie en float")

    # Pression
    if 'Pressure' in df.columns:
        df['Pressure'] = df['Pressure'].astype(str).apply(lambda x: float(re.sub(r'[^\d.]', '', x)))
        print(f"   Colonne 'Pressure' convertie en float")

    # Précipitations
    for col in ['Precip. Accum.', 'Precip. Rate.']:
        if col in df.columns:
            new_col = col.replace('.', '_').replace(' ', '_')
            df[new_col] = df[col].astype(str).apply(lambda x: float(re.sub(r'[^\d.]', '', x)))
            df = df.drop(columns=[col])
            print(f"   Colonne '{col}' nettoyée et renommée '{new_col}'")

    # UV et Solar
    for col in ['UV', 'Solar']:
        if col in df.columns:
            df[col] = df[col].astype(str).apply(lambda x: float(re.sub(r'[^\d.]', '', x)))
            print(f"   Colonne '{col}' convertie en float")

    # Time avec format précis
    if 'Time' in df.columns:
        df['Time'] = pd.to_datetime(df['Time'], format='%I:%M %p', errors='coerce')
        na_count = df['Time'].isna().sum()
        if na_count > 0:
            print(f" {na_count} valeurs NaT dans 'Time' seront remplacées par None")
        # Conversion finale en string ou None pour MongoDB
        df['Time'] = df['Time'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)

    # Wind reste texte
    if 'Wind' in df.columns:
        df['Wind'] = df['Wind'].astype(str).str.strip()
        print(f"   Colonne 'Wind' nettoyée (texte)")

    print(f" Nettoyage terminé pour {len(df)} lignes et {len(df.columns)} colonnes")
    return df

# Nettoyage des DataFrames
df_infoclimat = clean_weather_data(df_infoclimat)
df_ichtegem = clean_weather_data(df_ichtegem)
df_lamadeleine = clean_weather_data(df_lamadeleine)

# ===========================
# CELLULE 7 : Ajouter métadonnées stations amateurs
# ===========================
stations_metadata = [
    {
        "Station_ID": "ILAMAD25",
        "Name": "La Madeleine",
        "Latitude": 50.659,
        "Longitude": 3.07,
        "Elevation": 23,
        "City": "La Madeleine",
        "State": "-/-",
        "Hardware": "other",
        "Software": "EasyWeatherPro_V5.1.6"
    },
    {
        "Station_ID": "IICHTE19",
        "Name": "WeerstationBS",
        "Latitude": 51.092,
        "Longitude": 2.999,
        "Elevation": 15,
        "City": "Ichtegem",
        "State": "-/-",
        "Hardware": "other",
        "Software": "EasyWeatherV1.6.6"
    }
]

print("\n Métadonnées des stations définies")
# ===========================
# CELLULE 8 bis : Insertion des métadonnées stations
# ===========================
client = MongoClient("mongodb://localhost:27017/")
db = client["GreenAndCoop"]

print("\n📌 Connexion à MongoDB établie")
print("\n Insertion des métadonnées des stations amateurs")

db["Stations"].delete_many({})  # évite les doublons si on relance le script
db["Stations"].insert_many(stations_metadata)

print(f"   {len(stations_metadata)} stations insérées dans la collection 'Stations'")

# ===========================
# CELLULE 9 : Connexion MongoDB et insertion
# ===========================

db["InfoClimat"].insert_many(df_infoclimat.to_dict(orient="records"))
print(f"   {len(df_infoclimat)} lignes insérées dans InfoClimat")

db["WeatherIchtegem"].insert_many(df_ichtegem.to_dict(orient="records"))
print(f"   {len(df_ichtegem)} lignes insérées dans WeatherIchtegem")

db["WeatherLaMadeleine"].insert_many(df_lamadeleine.to_dict(orient="records"))
print(f"   {len(df_lamadeleine)} lignes insérées dans WeatherLaMadeleine")

print("\n Toutes les données ont été insérées avec succès dans MongoDB !")


 Librairies importées avec succès
AWS_ACCESS_KEY_ID chargé : True
AWS_SECRET_ACCESS_KEY chargé : True
 Connexion S3 établie pour le bucket greencoop-forecast2-raw
 Lecture du fichier JSONL : data_sync/InfoClimat_France/2026_01_12_1768220353795_0.jsonl
 Lecture terminée : 1 lignes et 6 colonnes
 Lecture du fichier JSONL : data_sync/weather_iichtegem/2026_01_12_1768224585495_0.jsonl
 Lecture terminée : 1899 lignes et 12 colonnes
 Lecture du fichier JSONL : data_sync/WeatherUnderground_LaMadeleine/2026_01_12_1768226046168_0.jsonl
 Lecture terminée : 1908 lignes et 12 colonnes

 Aperçu des données chargées :
InfoClimat : (1, 6)
Weather Ichtegem : (1899, 12)
Weather La Madeleine : (1908, 12)

 Début du nettoyage et transformation des données météo
 Nettoyage terminé pour 1 lignes et 6 colonnes

 Début du nettoyage et transformation des données météo
   Colonne 'Temperature' convertie en float
   Colonne 'Dew Point' convertie en float
   Colonne 'Humidity' convertie en float
   Colonne 'Spee